# Lesson 13 — Exercises

Run setup.sql first, and work through lesson.ipynb before attempting these. Use SSMS with the actual execution plan enabled (Ctrl+M) whenever a prompt asks about plan operators. Solutions with explanations: exercises-solutions.ipynb.

In [ ]:
USE AdventureWorks2022;
GO
SET STATISTICS IO   ON;
SET STATISTICS TIME ON;

## Exercise 1 — SARGable rewrite

Run the query below and look at its execution plan. Identify: (a) the plan operator on Sales.SalesOrderDetail, (b) estimated vs actual rows, (c) logical reads. Then rewrite the WHERE clause to be SARGable and compare.

In [ ]:
SELECT soh.SalesOrderID, SUM(sod.LineTotal) AS OrderTotal
FROM lesson13.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail    AS sod ON sod.SalesOrderID = soh.SalesOrderID
WHERE CONVERT(VARCHAR(7), soh.OrderDate, 120) = '2013-07'   -- non-SARGable
GROUP BY soh.SalesOrderID;

In [ ]:
-- Your SARGable rewrite here:

## Exercise 2 — eliminate a key lookup

The query below triggers a key lookup. Add an index to lesson13.SalesOrderHeader to eliminate it, then verify with the execution plan that the key lookup is gone.

In [ ]:
SELECT SalesOrderID, SalesPersonID, SubTotal
FROM lesson13.SalesOrderHeader
WHERE TerritoryID = 4;

In [ ]:
-- Your CREATE INDEX here:

## Exercise 3 — selectivity and access path

Run both queries and compare their plans and logical reads. Explain why the plans differ.

In [ ]:
-- Query A:
SELECT SalesOrderID, TotalDue FROM lesson13.SalesOrderHeader WHERE Status = 5;
-- Query B:
SELECT SalesOrderID, TotalDue FROM lesson13.SalesOrderHeader WHERE Status = 1;

**Your explanation here:** _(edit this cell)_

## Exercise 4 — parameter-sniffing-safe procedure

Create a stored procedure lesson13.usp_RevenueByTerritory that accepts @TerritoryID INT and returns SUM(TotalDue) grouped by YEAR(OrderDate). Call it with territory 1 (large) and territory 10 (small) and check whether the plan is appropriate for both. Add OPTION (OPTIMIZE FOR (@TerritoryID UNKNOWN)) if needed.

In [ ]:
-- Your CREATE PROCEDURE and test EXECs here:

## Exercise 5 — JOIN vs correlated subquery

Use the SET STATISTICS IO output to compare logical reads for the two queries below. Which one is more efficient, and why?

In [ ]:
-- Query A — explicit JOIN
SELECT p.Name, SUM(sod.OrderQty) AS TotalSold
FROM Sales.SalesOrderDetail     AS sod
JOIN Production.Product         AS p  ON p.ProductID = sod.ProductID
GROUP BY p.Name
ORDER BY TotalSold DESC;

In [ ]:
-- Query B — correlated subquery
SELECT p.Name,
       (SELECT SUM(OrderQty) FROM Sales.SalesOrderDetail sod WHERE sod.ProductID = p.ProductID) AS TotalSold
FROM Production.Product AS p
ORDER BY TotalSold DESC;

**Your analysis here:** _(edit this cell)_

In [ ]:
SET STATISTICS IO   OFF;
SET STATISTICS TIME OFF;